currently only used to extract the flagged outliers and inspect their feautures

In [ ]:
import numpy as np
import pandas as pd
import torch

from toy_thesis_minimal import simulate_toy_finrep, ToyConfig, DATE_COL, ID_COL
from inject_structural import inject_peer_anomaly
from feature_matrix import build_peer_ratio_matrix, compute_total_asset_ratios
from preprocessing import RobustIQRScaler
from models.vae import VAEDetector
from eval.metrics_clean import make_binary_labels # <-- needs a fix

# 1. Generate data and inject anomalies (as in the previous setup)
target_q = 202312
cfg = ToyConfig(n_banks=100, n_features=1000, n_factors=10)
clean_df, value_cols = simulate_toy_finrep(cfg)



###########################################################################################################################################change generating
injection = inject_peer_anomaly(
    clean_df, target_quarter=target_q, cols=value_cols,
    row_fraction=0.05, feature_fraction=0.10, random_state=42
)

# Separate history and target
historical_df = clean_df[clean_df[DATE_COL] < target_q].copy()
fm_train = compute_total_asset_ratios(historical_df, value_columns=value_cols)

corrupted_df = injection["data"]
fm_test = build_peer_ratio_matrix(corrupted_df, quarter=target_q, value_columns=value_cols)

# Scaling
scaler = RobustIQRScaler(imputation="median")
X_train_scaled = scaler.fit_transform(fm_train.X)
X_test_scaled = scaler.transform(fm_test.X)

# 2. Train VAE
vae = VAEDetector(latent_dim=10, hidden_dim=32, epochs=500, beta=0.1, random_state=42)
vae.fit(X_train_scaled)

# 3. Calculate anomaly scores and flag the top X outliers
scores = vae.score_samples(X_test_scaled)

#  build analysis DF
analysis_df = fm_test.metadata.copy()
analysis_df["anomaly_score"] = scores

analysis_df["is_true_anomaly"] = make_binary_labels(fm_test.metadata, injection["row_labels"], outlier_type="peer")

# Sort by score
analysis_df = analysis_df.sort_values(by="anomaly_score", ascending=False)

#  take  top 5  banks
top_anomalies = analysis_df.head(5)
print("Top 5 flagged anomalies:")
display(top_anomalies[[ID_COL, "anomaly_score", "is_true_anomaly"]])

# 4. Feature attribution: 
top_k_features = 5 #  driver cells per bank
X_arr = X_test_scaled.astype("float32").to_numpy()

# reconstructions from  VAE PyTorch model
vae.model_.eval()
with torch.no_grad():
    x_tensor = torch.tensor(X_arr)
    mu, _ = vae.model_.encode(x_tensor)
    reconstructions = vae.model_.decode(mu).numpy()

# Cell-specific squared error: (X - X_hat)^2
squared_residuals = (X_arr - reconstructions) ** 2
feature_names = np.array(X_test_scaled.columns)

# Get ground truth for cells 
ground_truth_rows = injection["row_labels"]
ground_truth_cells = injection.get("cell_labels", None)

print(f"\n--- Root Cause Analysis: Top {top_k_features} driving cells per bank ---")

for idx, row in top_anomalies.iterrows():
    bank_id = row[ID_COL]
    total_score = row["anomaly_score"]
    is_true = row["is_true_anomaly"]
    
    # Get residuals for  this row
    row_residuals = squared_residuals[idx]
    
    # Find  indices 
    top_feature_indices = np.argsort(row_residuals)[-top_k_features:][::-1]
    
    print(f"\nBank: {bank_id} | Total Score: {total_score:.2f} | True Anomaly: {bool(is_true)}")
    
    predicted_features = set()
    
    for rank, feat_idx in enumerate(top_feature_indices, 1):
        feat_name = feature_names[feat_idx]
        cell_error = row_residuals[feat_idx]
        original_val = X_test_scaled.iloc[idx, feat_idx]
        recon_val = reconstructions[idx, feat_idx]
        
        # For the ground truth match: clean up "ratio__" prefix
        clean_feat_name = str(feat_name).replace("ratio__", "").replace("delta_ratio__", "")
        predicted_features.add(clean_feat_name)
        
        print(f"  {rank}. {feat_name}: Error = {cell_error:.2f} "
              f"(Scaled: Original={original_val:.2f}, Model expected={recon_val:.2f})")

    # Do I need
    if is_true == 1 and ground_truth_cells is not None:
        # Find the correct row in cell_labels 
        gt_row_idx = ground_truth_rows[ground_truth_rows[ID_COL] == bank_id].index[0]
        gt_cells = ground_truth_cells.loc[gt_row_idx]
        
        #  cells marked as manipulated 
        actual_features = set(gt_cells[gt_cells != "none"].index)
        hits = predicted_features.intersection(actual_features)
        
        print(f"  -> Check: {len(actual_features)} cells were manipulated.")
        print(f"  -> Check: {len(hits)} of them were found in the top {top_k_features}.")
        if hits:
            print(f"  -> Exact hits: {hits}")


#######################global
if ground_truth_cells is not None:
    true_anomalies = analysis_df[analysis_df["is_true_anomaly"] == 1].copy()
    
    total_hits = 0
    total_injected = 0
    
    for idx, row in true_anomalies.iterrows():
        bank_id = row[ID_COL]
        
        # VAE error for  row
        row_residuals = squared_residuals[idx]
        top_feature_indices = np.argsort(row_residuals)[-top_k_features:][::-1]
        predicted_features = set([str(feature_names[i]).replace("ratio__", "") for i in top_feature_indices])
        
        # ground truth for  row
        gt_row_idx = ground_truth_rows[ground_truth_rows[ID_COL] == bank_id].index[0]
        gt_cells = ground_truth_cells.loc[gt_row_idx]
        actual_features = set(gt_cells[gt_cells != "none"].index)
        
        hits = predicted_features.intersection(actual_features)
        total_hits += len(hits)
        total_injected += len(actual_features)

    if total_injected > 0:
        global_hit_rate = total_hits / total_injected
        print("\n" + "="*65)
        print(f"GLOBAL CELL DETECTION RATE (Recall @ k={top_k_features}): {global_hit_rate:.2%}")
        print("="*65)

Top 5 flagged anomalies:


,gebernummer,anomaly_score,is_true_anomaly
40,TOY000041,14.355888,0
8,TOY000009,13.926126,1
12,TOY000013,13.058841,0
75,TOY000076,12.485372,1
99,TOY000100,12.435660,1



--- Root Cause Analysis: Top 5 driving cells per bank ---

Bank: TOY000041 | Total Score: 14.36 | True Anomaly: False
  1. ratio__F_TOY_0231: Error = 4.96 (Scaled: Original=0.00, Model expected=2.23)
  2. ratio__F_TOY_0499: Error = 2.44 (Scaled: Original=0.00, Model expected=1.56)
  3. ratio__F_TOY_0497: Error = 1.13 (Scaled: Original=0.00, Model expected=1.06)
  4. ratio__F_TOY_0420: Error = 1.11 (Scaled: Original=0.00, Model expected=1.05)
  5. ratio__F_TOY_0317: Error = 1.10 (Scaled: Original=2.00, Model expected=0.95)

Bank: TOY000009 | Total Score: 13.93 | True Anomaly: True
  1. ratio__F_TOY_0582: Error = 8.68 (Scaled: Original=-2.01, Model expected=0.93)
  2. ratio__F_TOY_0804: Error = 5.70 (Scaled: Original=-1.44, Model expected=0.94)
  3. ratio__F_TOY_0805: Error = 5.39 (Scaled: Original=-1.38, Model expected=0.94)
  4. ratio__F_TOY_0391: Error = 4.22 (Scaled: Original=-1.06, Model expected=1.00)
  5. ratio__F_TOY_0122: Error = 3.89 (Scaled: Original=-0.62, Model expected=1.3